# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

**Học viên:** Bùi Văn Quang — **MSSV:** 2A202602688  
**Track 4 · Ngày 1 · VinUniversity AICB 2026**  
Bài toán: Phân loại loại rừng Forest CoverType (7 lớp, 54 đặc trưng).

In [ ]:
# ===== 1. Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch

# Kiểm tra thư mục chạy: nếu đang trong code/ thì REPO_ROOT='../..' và OUT_DIR='..'
# Nếu đang ở thư mục gốc thì chỉnh linh hoạt
if Path("../../scripts").exists():
    REPO_ROOT = "../.."
    OUT_DIR = ".."
elif Path("scripts").exists():
    REPO_ROOT = "."
    OUT_DIR = "submission_2A202602688"
else:
    REPO_ROOT = "."
    OUT_DIR = "."

# Tạo thư mục figures và results
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

# Thiết bị tính toán: ưu tiên CUDA, sau đó MPS (Mac), cuối cùng CPU
if torch.cuda.is_available():
    device = "cuda"
    dev_name = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    dev_name = "Apple Silicon MPS"
else:
    device = "cpu"
    dev_name = "CPU"

print(f"PyTorch Version : {torch.__version__}")
print(f"Device Sử Dụng   : {device} ({dev_name})")
print(f"REPO_ROOT       : {Path(REPO_ROOT).resolve()}")
print(f"OUT_DIR         : {Path(OUT_DIR).resolve()}")

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
if str(Path(REPO_ROOT) / "code") not in sys.path:
    sys.path.append(str(Path(REPO_ROOT) / "code"))

from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, compute_loss, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Chuẩn bị và kiểm tra dữ liệu
- Chạy script chia dữ liệu `scripts/split_data.py` (nếu chưa có `train.npz`, `eval.npz`)
- Tách 20% validation từ tập train (phân tầng theo nhãn, seed 42)
- Chuẩn hoá 10 đặc trưng liên tục chỉ bằng thống kê train
- Đưa toàn bộ tensor lên device

In [ ]:
proc_dir = Path(REPO_ROOT) / "data" / "processed"
if not (proc_dir / "train.npz").exists() or not (proc_dir / "eval.npz").exists():
    print("Chưa có dữ liệu processed, đang chạy scripts/split_data.py...")
    subprocess.run([sys.executable, str(Path(REPO_ROOT) / "scripts" / "split_data.py")], cwd=REPO_ROOT, check=True)

data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=str(proc_dir))

# Kiểm tra shape và kiểu dữ liệu
print("Shape tensor train :", data["X_tr"].shape, data["y_tr"].shape)
print("Shape tensor val   :", data["X_val"].shape, data["y_val"].shape)
print("Shape tensor eval  :", data["X_eval"].shape, data["y_eval"].shape)

# Kiểm tra chuẩn hoá trên tập train con
numeric_means = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
numeric_stds = data["X_tr"][:, :10].std(dim=0).cpu().numpy()
print("Mean 10 cột số (train):", np.round(numeric_means, 4))
print("Std 10 cột số (train) :", np.round(numeric_stds, 4))
assert np.allclose(numeric_means, 0.0, atol=1e-2), "Mean chuẩn hoá lệch quá lớn so với 0!"
assert np.allclose(numeric_stds, 1.0, atol=1e-2), "Std chuẩn hoá lệch quá lớn so với 1!"

## Part 1 — Model và các phép thử "sức khoẻ" ban đầu (Chương 5 slide)
1. Khởi tạo `M-base` `(54 → 256 → 128 → 7)`: assert 47 879 tham số.
2. Cho lô ngẫu nhiên `(8, 54)` chạy qua kiểm tra shape logits `(8, 7)`.
3. Đo loss bước 0 trên val ở eval mode: so sánh với kỳ vọng $\ln 7 \approx 1.9459$.
4. Quá khớp 20 mẫu (overfit 20 samples): loss phải giảm về gần 0 (accuracy 100%).
5. Kiểm tra gradient chảy: mọi tham số phải có `grad != None` và khác 0.

In [ ]:
set_seed(42)

# 1. Khởi tạo model và assert số tham số
model_check = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
p_count = count_params(model_check)
print(f"Số tham số của M-base: {p_count:,d}")
assert p_count == EXPECTED_PARAMS[(256, 128)], f"Số tham số {p_count} không đúng chuẩn 47,879!"

# 2. Kiểm tra shape logits
x_dummy = torch.randn(8, 54, device=device)
logits_dummy = model_check(x_dummy)
print(f"Shape logits đầu ra của lô (8, 54): {logits_dummy.shape}")
assert logits_dummy.shape == (8, 7), "Shape logits không khớp (8, 7)!"

# 3. Loss bước 0 trên val
step0_eval = evaluate(model_check, data["X_val"], data["y_val"], loss_name="ce")
ln7 = np.log(7.0)
print(f"Loss bước 0 đo được: {step0_eval['loss']:.4f} (Kỳ vọng ln(7) = {ln7:.4f}, chênh lệch: {abs(step0_eval['loss'] - ln7):.4f})")

# 4. Quá khớp 20 mẫu
x_tiny = data["X_tr"][:20]
y_tiny = data["y_tr"][:20]
opt_tiny = torch.optim.Adam(model_check.parameters(), lr=0.01)
tiny_losses = []
for step in range(300):
    opt_tiny.zero_grad()
    out = model_check(x_tiny)
    loss = compute_loss(out, y_tiny, "ce")
    loss.backward()
    opt_tiny.step()
    tiny_losses.append(loss.item())

final_tiny_loss = tiny_losses[-1]
final_tiny_acc = (model_check(x_tiny).argmax(dim=1) == y_tiny).float().mean().item()
print(f"Quá khớp 20 mẫu sau 300 bước: Loss = {final_tiny_loss:.6f}, Accuracy = {final_tiny_acc*100:.1f}%")
assert final_tiny_loss < 0.05 and final_tiny_acc == 1.0, "Không quá khớp được 20 mẫu!"

# 5. Kiểm tra gradient chảy qua tất cả các tham số
model_check.zero_grad()
dummy_loss = compute_loss(model_check(x_tiny[:2]), y_tiny[:2], "ce")
dummy_loss.backward()
print("Kiểm tra gradient các tầng tham số:")
for name, param in model_check.named_parameters():
    assert param.grad is not None, f"Gradient của {name} bị None!"
    gn = param.grad.norm().item()
    print(f"  - {name:20s}: shape={str(tuple(param.shape)):15s} | grad_norm={gn:.6f}")
    assert gn > 0.0, f"Gradient của {name} bằng 0 (gradient bị tắt)!"

**Nhận xét Part 1:**
- Model chuẩn xác 47 879 tham số, logits `(B, 7)`.
- Loss bước 0 rất sát với $\ln 7 \approx 1.946$, chứng tỏ trọng số khởi tạo He cân bằng, không có lớp nào bị thiên vị.
- Mô hình overfit hoàn toàn 20 mẫu (Loss $\to 0$, Accuracy = 100%), xác nhận pipeline autograd, backward và optimizer hoạt động chính xác.
- Mọi tensor trọng số và bias đều có gradient khác 0, không có nơ-ron nào bị chết hay tắc gradient.

## Part 2 — Huấn luyện Baseline và đo độ nhiễu giữa các Seed
- Baseline: kiến trúc `M-base`, He initialization, hàm mất mát Cross-Entropy, bộ tối ưu SGD + momentum 0.9, batch size 512, 20 epoch.
- Chạy với 3 seed khác nhau (1, 2, 3) để đo độ lệch chuẩn $\sigma$ và xác định ngưỡng nhiễu $2\sigma$ của `val_macro_f1`.

In [ ]:
# Khảo sát chọn learning rate hợp lý cho SGD+momentum trên tập val
lr_candidates = [0.01, 0.05, 0.1]
print("Khảo sát nhanh lr cho baseline (chạy 5 epoch trên val)...")
best_lr = 0.05
best_lr_score = -1.0
for test_lr in lr_candidates:
    quick_cfg = {**DEFAULT_CFG, "lr": test_lr, "epochs": 5, "exp_id": f"lr-tuning-{test_lr}"}
    res_quick = run_experiment(quick_cfg, data)
    score = res_quick["summary"]["val_macro_f1"]
    print(f"  lr = {test_lr:5.2f} -> val_macro_f1 = {score:.4f}")
    if score > best_lr_score:
        best_lr_score = score
        best_lr = test_lr

print(f"=> Chọn lr tối ưu cho baseline = {best_lr}")
base_cfg = {**DEFAULT_CFG, "lr": best_lr, "epochs": 20}

# Chạy baseline với 3 seed: seed 1, 2, 3
baseline_results = []
for s in [1, 2, 3]:
    cfg_s = {**base_cfg, "seed": s, "exp_id": f"base-s{s}", "description": f"Baseline He SGD+M lr={best_lr} seed={s}"}
    print(f"\n--- Đang chạy {cfg_s['exp_id']} ---")
    res_s = run_experiment(cfg_s, data)
    baseline_results.append(res_s)
    save_result(res_s, f"{OUT_DIR}/results")
    plot_run(res_s, f"{OUT_DIR}/figures/{cfg_s['exp_id']}.png")
    sm = res_s["summary"]
    print(f"  {cfg_s['exp_id']}: Best Val Loss={sm['best_val_loss']:.4f} (ep {sm['best_epoch']}), Val Acc={sm['val_acc']:.4f}, Val Macro-F1={sm['val_macro_f1']:.4f}")

# Thống kê độ nhiễu seed
base_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]
base_accs = [r["summary"]["val_acc"] for r in baseline_results]
f1_mean, f1_std = float(np.mean(base_f1s)), float(np.std(base_f1s, ddof=1))
acc_mean, acc_std = float(np.mean(base_accs)), float(np.std(base_accs, ddof=1))
noise_threshold_2sigma = 2.0 * f1_std

print(f"\n=== KẾT QUẢ ĐỘ NHIỄU BASELINE (3 SEEDS) ===")
print(f"Val Accuracy : {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Val Macro-F1 : {f1_mean:.4f} ± {f1_std:.4f}")
print(f"Ngưỡng nhiễu (2σ): {noise_threshold_2sigma:.4f}")

## Part 3 — Thí nghiệm các yếu tố ảnh hưởng (Đầy đủ 7 chủ đề)
Tiến hành thí nghiệm trên từng yếu tố (mỗi lần chỉ đổi 1 yếu tố so với baseline, giữ nguyên seed 1, số epoch 20 và split):
1. **Chủ đề 1 — Loss:** `loss-mse` (CE vs MSE trên one-hot)
2. **Chủ đề 2 — Optimizer:** `opt-sgd` (SGD thuần), `opt-adam-lr1e-3`, `opt-adam-lr3e-4`, `opt-adamw`
3. **Chủ đề 3 — Hyper-parameters:** `hparam-batch128`, `hparam-batch2048`, `hparam-mwide`, `hparam-mdeep`
4. **Chủ đề 4 — Dropout:** `drop-0.2`, `drop-0.5`
5. **Chủ đề 5 — Gradient Clipping:** `clip-1.0` (ở lr chuẩn) và `clip-highlr-1.0` vs `noclip-highlr` (lr=0.5)
6. **Chủ đề 6 — Mixed Precision:** `amp-fp16`
7. **Chủ đề 7 — Khởi tạo:** `init-zeros`, `init-normal`, `init-xavier`

In [ ]:
all_experiments = {}
# Lưu baseline s1
all_experiments["base-s1"] = baseline_results[0]

# Danh sách cấu hình thí nghiệm được thiết kế chuẩn xác
experiment_configs = [
    # 1. Loss
    {
        "exp_id": "loss-mse", "group": "loss",
        "description": "Hàm mất mát MSE trên one-hot",
        "loss": "mse", "optimizer": "sgd_momentum", "lr": best_lr,
    },
    # 2. Optimizer
    {
        "exp_id": "opt-sgd", "group": "optimizer",
        "description": "SGD thuần không momentum",
        "optimizer": "sgd", "lr": best_lr,
    },
    {
        "exp_id": "opt-adam-lr1e-3", "group": "optimizer",
        "description": "Adam với lr=0.001",
        "optimizer": "adam", "lr": 1e-3,
    },
    {
        "exp_id": "opt-adam-lr3e-4", "group": "optimizer",
        "description": "Adam với lr=0.0003",
        "optimizer": "adam", "lr": 3e-4,
    },
    {
        "exp_id": "opt-adamw", "group": "optimizer",
        "description": "AdamW lr=1e-3 weight_decay=0.01",
        "optimizer": "adamw", "lr": 1e-3, "weight_decay": 0.01,
    },
    # 3. Hyper-parameters
    {
        "exp_id": "hparam-batch128", "group": "hparam",
        "description": "Batch size nhỏ 128 (nhiều bước cập nhật hơn)",
        "batch": 128, "optimizer": "sgd_momentum", "lr": best_lr,
    },
    {
        "exp_id": "hparam-batch2048", "group": "hparam",
        "description": "Batch size lớn 2048 (ít bước cập nhật hơn)",
        "batch": 2048, "optimizer": "sgd_momentum", "lr": best_lr,
    },
    {
        "exp_id": "hparam-mwide", "group": "hparam",
        "description": "Kiến trúc M-wide (54->512->256->7)",
        "hidden": (512, 256), "optimizer": "sgd_momentum", "lr": best_lr,
    },
    {
        "exp_id": "hparam-mdeep", "group": "hparam",
        "description": "Kiến trúc M-deep (54->256->128->64->7)",
        "hidden": (256, 128, 64), "optimizer": "sgd_momentum", "lr": best_lr,
    },
    # 4. Dropout
    {
        "exp_id": "drop-0.2", "group": "dropout",
        "description": "Dropout q=0.2 sau ReLU lớp ẩn",
        "dropout": 0.2, "optimizer": "sgd_momentum", "lr": best_lr,
    },
    {
        "exp_id": "drop-0.5", "group": "dropout",
        "description": "Dropout q=0.5 sau ReLU lớp ẩn",
        "dropout": 0.5, "optimizer": "sgd_momentum", "lr": best_lr,
    },
    # 5. Gradient Clipping
    {
        "exp_id": "clip-1.0", "group": "clipping",
        "description": "Cắt gradient max_norm=1.0 ở lr chuẩn",
        "clip_norm": 1.0, "optimizer": "sgd_momentum", "lr": best_lr,
    },
    {
        "exp_id": "clip-highlr-1.0", "group": "clipping",
        "description": "lr cao 0.5 có gradient clipping 1.0",
        "lr": 0.5, "clip_norm": 1.0, "optimizer": "sgd_momentum",
    },
    {
        "exp_id": "noclip-highlr", "group": "clipping",
        "description": "lr cao 0.5 KHÔNG clipping (phản chứng bùng nổ gradient)",
        "lr": 0.5, "clip_norm": None, "optimizer": "sgd_momentum",
    },
    # 6. Mixed Precision
    {
        "exp_id": "amp-fp16", "group": "amp",
        "description": "Mixed precision FP16 với GradScaler",
        "precision": "fp16" if torch.cuda.is_available() else "fp32",
        "optimizer": "sgd_momentum", "lr": best_lr,
    },
    # 7. Khởi tạo tham số
    {
        "exp_id": "init-zeros", "group": "init",
        "description": "Khởi tạo W=0, b=0 (mất đối xứng)",
        "init": "zeros", "optimizer": "sgd_momentum", "lr": best_lr,
    },
    {
        "exp_id": "init-normal", "group": "init",
        "description": "Khởi tạo W ~ N(0, 0.01^2)",
        "init": "normal", "optimizer": "sgd_momentum", "lr": best_lr,
    },
    {
        "exp_id": "init-xavier", "group": "init",
        "description": "Khởi tạo Xavier Normal",
        "init": "xavier", "optimizer": "sgd_momentum", "lr": best_lr,
    },
]

# Chạy tuần tự và thu thập kết quả
for item in experiment_configs:
    cfg = {**base_cfg, **item}
    print(f"\n>>> Chạy Thí Nghiệm [{cfg['group']}]: {cfg['exp_id']} - {cfg['description']}")
    res = run_experiment(cfg, data)
    all_experiments[cfg["exp_id"]] = res
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")
    
    sm = res["summary"]
    delta_f1 = sm["val_macro_f1"] - base_f1s[0]
    beyond = abs(delta_f1) > noise_threshold_2sigma
    print(f"    Val Macro-F1 = {sm['val_macro_f1']:.4f} (Delta vs base-s1: {delta_f1:+.4f}) | Vượt nhiễu 2σ ({noise_threshold_2sigma:.4f}): {'CÓ' if beyond else 'KHÔNG'}")

### Vẽ ảnh so sánh theo từng nhóm chủ đề (`compare_<nhóm>.png`)

In [ ]:
groups = ["optimizer", "hparam", "dropout", "clipping", "init"]
for grp in groups:
    group_runs = [res for res in all_experiments.values() if res["cfg"].get("group") == grp or res["cfg"].get("exp_id") == "base-s1"]
    if len(group_runs) > 1:
        plot_compare(group_runs, "val_loss", f"{OUT_DIR}/figures/compare_{grp}_loss.png", title=f"So sánh Val Loss - Nhóm {grp}")
        plot_compare(group_runs, "val_macro_f1", f"{OUT_DIR}/figures/compare_{grp}.png", title=f"So sánh Val Macro-F1 - Nhóm {grp}")
        print(f"Đã vẽ ảnh so sánh nhóm {grp} vào {OUT_DIR}/figures/compare_{grp}.png")

## Part 4 — Đánh giá cuối trên tập Eval, Phân tích Lỗi và Xuất Bảng
1. **Chọn cấu hình tốt nhất dựa trên Validation**: So sánh điểm `val_macro_f1` giữa các thí nghiệm.
2. Huấn luyện cấu hình cuối cùng (Final Model) và chạy `final_eval` tạo `predictions_eval.csv`.
3. Chạy `scripts/evaluate.py` chính thức để tạo `eval_result.json`.
4. Phân tích chi tiết ma trận nhầm lẫn và F1 từng lớp.
5. Xuất bảng tổng hợp `experiments.xlsx` từ template.

In [ ]:
# Tìm cấu hình có val_macro_f1 cao nhất trong các thí nghiệm
best_exp_id = max(all_experiments.keys(), key=lambda k: all_experiments[k]["summary"]["val_macro_f1"])
best_exp = all_experiments[best_exp_id]
print(f"Cấu hình tốt nhất trên Validation là: {best_exp_id}")
print(f"Mô tả: {best_exp['cfg'].get('description')}")
print(f"Val Acc: {best_exp['summary']['val_acc']:.4f}, Val Macro-F1: {best_exp['summary']['val_macro_f1']:.4f}")

# Dự đoán trên tập eval cho cấu hình tốt nhất
pred_eval_path = f"{OUT_DIR}/predictions_eval.csv"
final_eval(best_exp["cfg"], best_exp, data, pred_eval_path)

# Đánh giá chính thức bằng scripts/evaluate.py
eval_json_path = f"{OUT_DIR}/eval_result.json"
eval_cmd = [
    sys.executable, str(Path(REPO_ROOT) / "scripts" / "evaluate.py"),
    "--pred", pred_eval_path,
    "--data", str(Path(REPO_ROOT) / "data" / "covtype.csv.gz"),
    "--meta", str(Path(REPO_ROOT) / "data" / "split_metadata.csv"),
    "--out", eval_json_path
]
proc = subprocess.run(eval_cmd, capture_output=True, text=True, check=True)
print(proc.stdout)

# Đánh giá eval cho Baseline để so sánh
pred_base_path = f"{OUT_DIR}/predictions_base.csv"
eval_base_json_path = f"{OUT_DIR}/eval_base_result.json"
final_eval(baseline_results[0]["cfg"], baseline_results[0], data, pred_base_path)
eval_base_cmd = [
    sys.executable, str(Path(REPO_ROOT) / "scripts" / "evaluate.py"),
    "--pred", pred_base_path,
    "--data", str(Path(REPO_ROOT) / "data" / "covtype.csv.gz"),
    "--meta", str(Path(REPO_ROOT) / "data" / "split_metadata.csv"),
    "--out", eval_base_json_path
]
subprocess.run(eval_base_cmd, capture_output=True, text=True, check=True)

with open(eval_json_path, "r", encoding="utf-8") as f:
    eval_final_res = json.load(f)
with open(eval_base_json_path, "r", encoding="utf-8") as f:
    eval_base_res = json.load(f)

print("=== KẾT QUẢ SO SÁNH EVAL CUỐI CÙNG ===")
print(f"Baseline (base-s1) : Accuracy = {eval_base_res['accuracy']:.4f} | Eval Macro-F1 = {eval_base_res['macro_f1']:.4f}")
print(f"Final ({best_exp_id}) : Accuracy = {eval_final_res['accuracy']:.4f} | Eval Macro-F1 = {eval_final_res['macro_f1']:.4f}")
print(f"Cải thiện Macro-F1 : {eval_final_res['macro_f1'] - eval_base_res['macro_f1']:+.4f}")

### Phân tích lỗi theo từng lớp trên tập Eval

In [ ]:
per_class_df = pd.DataFrame(eval_final_res["per_class"])
class_names = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]
per_class_df["Tên Lớp"] = class_names
per_class_df = per_class_df[["cls", "Tên Lớp", "support", "precision", "recall", "f1"]]
print(per_class_df.to_string(index=False))

# In ma trận nhầm lẫn
cm_df = pd.DataFrame(eval_final_res["confusion_matrix"], index=class_names, columns=class_names)
print("\nMa trận nhầm lẫn:")
print(cm_df)

### Xuất file Excel `experiments.xlsx` từ Template

In [ ]:
loaded_results = load_results(f"{OUT_DIR}/results")
rows = []
for r in loaded_results:
    exp_id = r["cfg"]["exp_id"]
    eval_scores = None
    if exp_id == "base-s1":
        eval_scores = {"accuracy": eval_base_res["accuracy"], "macro_f1": eval_base_res["macro_f1"]}
    elif exp_id == best_exp_id:
        eval_scores = {"accuracy": eval_final_res["accuracy"], "macro_f1": eval_final_res["macro_f1"]}
    rows.append(to_row(r, eval_scores=eval_scores))

template_xlsx = Path(REPO_ROOT) / "templates" / "experiment_table_template.xlsx"
out_xlsx = Path(OUT_DIR) / "experiments.xlsx"
write_xlsx(rows, str(template_xlsx), str(out_xlsx))
print(f"Đã xuất bảng tổng hợp thí nghiệm thành công ra {out_xlsx}!")